# 🍷 From CSV to RDF Knowledge Graph — Wine Edition

Knowledge Graphs are powerful structures that represent data as entities and relationships rather than flat tables.  
Instead of thinking in rows and columns, a knowledge graph connects information in a *semantic network* — making it easy to reason about relationships and retrieve meaningful patterns.

In this tutorial, we'll:
1. Load the **Wine Reviews dataset** from Kaggle.
2. Design a simple **RDF schema** (Wine, Winery, Variety, Region, etc.).
3. Convert the tabular data into **RDF triples** using Python’s [`rdflib`](https://rdflib.readthedocs.io/).
4. Serialize the graph into Turtle format (`.ttl`) and run SPARQL queries on it.

By the end, you’ll have a fully working **Wine Knowledge Graph** you can extend, visualize, or even use in a future **GraphRAG pipeline** 🍇


In [45]:
from rdflib import Graph, Literal, RDF, URIRef, Namespace, XSD
from rdflib.namespace import RDFS, OWL, DC, FOAF
from datetime import date
from tqdm import tqdm
import pandas as pd
DATA_FOLDER = "../data/"
INPUT_DATA_PATH = f"{DATA_FOLDER}winemag-data_first150k.csv"
OUTPUT_FOLDER = f"{DATA_FOLDER}output/"

In [43]:
# Load the Wine Reviews dataset (downloaded from Kaggle into the data folder)

df = pd.read_csv(INPUT_DATA_PATH, index_col=0)
print("✅ Data loaded successfully.")

# Explore the dataset
print(df.describe(),df.info())
df.head()


✅ Data loaded successfully.
<class 'pandas.core.frame.DataFrame'>
Index: 150930 entries, 0 to 150929
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   country      150925 non-null  object 
 1   description  150930 non-null  object 
 2   designation  105195 non-null  object 
 3   points       150930 non-null  int64  
 4   price        137235 non-null  float64
 5   province     150925 non-null  object 
 6   region_1     125870 non-null  object 
 7   region_2     60953 non-null   object 
 8   variety      150930 non-null  object 
 9   winery       150930 non-null  object 
dtypes: float64(1), int64(1), object(8)
memory usage: 12.7+ MB
              points          price
count  150930.000000  137235.000000
mean       87.888418      33.131482
std         3.222392      36.322536
min        80.000000       4.000000
25%        86.000000      16.000000
50%        88.000000      24.000000
75%        90.000000      40.000000

,country,description,designation,points,price,province,region_1,region_2,variety,winery
0,US,This tremendous 100% varietal wine hails from ...,Martha's Vineyard,96,235.0,California,Napa Valley,Napa,Cabernet Sauvignon,Heitz
1,Spain,"Ripe aromas of fig, blackberry and cassis are ...",Carodorum Selección Especial Reserva,96,110.0,Northern Spain,Toro,NaN,Tinta de Toro,Bodega Carmen Rodríguez
2,US,Mac Watson honors the memory of a wine once ma...,Special Selected Late Harvest,96,90.0,California,Knights Valley,Sonoma,Sauvignon Blanc,Macauley
3,US,"This spent 20 months in 30% new French oak, an...",Reserve,96,65.0,Oregon,Willamette Valley,Willamette Valley,Pinot Noir,Ponzi
4,France,"This is the top wine from La Bégude, named aft...",La Brûlade,95,66.0,Provence,Bandol,NaN,Provence red blend,Domaine de la Bégude


In [33]:
# Initialize the main knowledge graph
g = Graph()

# Initialize metadata graph
g_meta = Graph()

# --- Namespaces ---
WINE = Namespace("https://wafaahs.github.io/wine-kg#")
META = Namespace("https://wafaahs.github.io/wine-kg/meta#")

# --- Bind prefixes ---
for graph in [g, g_meta]:
    graph.bind("wine", WINE)
    graph.bind("meta", META)
    graph.bind("rdfs", RDFS)
    graph.bind("owl", OWL)
    graph.bind("dc", DC)
    graph.bind("foaf", FOAF)
    graph.bind("xsd", XSD)

print("✅ RDF graphs initialized with custom namespaces.")

✅ RDF graphs initialized with custom namespaces.


In [34]:
# Define high-level classes for ontology clarity
for cls in ["Wine", "Variety", "Winery", "Region", "Province", "Country", "Review", "Taster", "Designation"]:
    g.add((WINE[cls], RDF.type, OWL.Class))

In [35]:
today = date.today().isoformat()

for i, row in tqdm(df.head(2000).iterrows(), total=min(len(df), 2000)):
    wine_uri = URIRef(WINE[f"Wine_{i}"])
    g.add((wine_uri, RDF.type, WINE.Wine))

    # --- Track missing fields for metadata ---
    missing_fields = [col for col in df.columns if pd.isna(row[col])]
    if missing_fields:
        for field in missing_fields:
            g_meta.add((wine_uri, META.hasMissingField, Literal(field)))
        g_meta.add((wine_uri, META.missingCount, Literal(len(missing_fields), datatype=XSD.integer)))
    g_meta.add((wine_uri, META.processedOn, Literal(today, datatype=XSD.date)))
    g_meta.add((wine_uri, META.sourceRow, Literal(i, datatype=XSD.integer)))

    # --- Variety ---
    if pd.notna(row.get("variety")):
        variety_uri = URIRef(WINE[row["variety"].replace(" ", "_")])
        g.add((variety_uri, RDF.type, WINE.Variety))
        g.add((wine_uri, WINE.hasVariety, variety_uri))

    # --- Winery ---
    if pd.notna(row.get("winery")):
        winery_uri = URIRef(WINE[row["winery"].replace(" ", "_")])
        g.add((winery_uri, RDF.type, WINE.Winery))
        g.add((wine_uri, WINE.producedBy, winery_uri))

    # --- Designation (specific label) ---
    if pd.notna(row.get("designation")):
        desig_uri = URIRef(WINE[row["designation"].replace(" ", "_")])
        g.add((desig_uri, RDF.type, WINE.Designation))
        g.add((wine_uri, WINE.hasDesignation, desig_uri))

    # --- Geography hierarchy ---
    region2_uri = region1_uri = province_uri = country_uri = None

    if pd.notna(row.get("region_2")):
        region2_uri = URIRef(WINE[row["region_2"].replace(" ", "_")])
        g.add((region2_uri, RDF.type, WINE.Region))
        g.add((wine_uri, WINE.fromSubregion, region2_uri))

    if pd.notna(row.get("region_1")):
        region1_uri = URIRef(WINE[row["region_1"].replace(" ", "_")])
        g.add((region1_uri, RDF.type, WINE.Region))
        g.add((wine_uri, WINE.fromRegion, region1_uri))
        if region2_uri:
            g.add((region2_uri, WINE.inRegion, region1_uri))

    if pd.notna(row.get("province")):
        province_uri = URIRef(WINE[row["province"].replace(" ", "_")])
        g.add((province_uri, RDF.type, WINE.Province))
        if region1_uri:
            g.add((region1_uri, WINE.inProvince, province_uri))

    if pd.notna(row.get("country")):
        country_uri = URIRef(WINE[row["country"].replace(" ", "_")])
        g.add((country_uri, RDF.type, WINE.Country))
        if province_uri:
            g.add((province_uri, WINE.inCountry, country_uri))
        elif region1_uri:
            g.add((region1_uri, WINE.inCountry, country_uri))
        elif region2_uri:
            g.add((region2_uri, WINE.inCountry, country_uri))
        else:
            g.add((wine_uri, WINE.fromCountry, country_uri))

    # --- Points and Price ---
    if pd.notna(row.get("points")):
        g.add((wine_uri, WINE.points, Literal(int(row["points"]), datatype=XSD.integer)))
    if pd.notna(row.get("price")):
        g.add((wine_uri, WINE.price, Literal(float(row["price"]), datatype=XSD.decimal)))

    # --- Description ---
    if pd.notna(row.get("description")):
        review_uri = URIRef(WINE[f"Review_{i}"])
        g.add((review_uri, RDF.type, WINE.Review))
        g.add((review_uri, WINE.aboutWine, wine_uri))
        g.add((review_uri, WINE.description, Literal(row["description"])))


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:01<00:00, 1362.63it/s]


In [46]:
# Serialize main graph
main_path = f"{OUTPUT_FOLDER}wine_graph.ttl"
g.serialize(destination=main_path, format="turtle")

# Serialize metadata graph
meta_path = f"{OUTPUT_FOLDER}wine_metadata.ttl"
g_meta.serialize(destination=meta_path, format="turtle")

print(f"✅ Main graph saved to {main_path} ({len(g)} triples)")
print(f"✅ Metadata graph saved to {meta_path} ({len(g_meta)} triples)")


✅ Main graph saved to ../data/output/wine_graph.ttl (23679 triples)
✅ Metadata graph saved to ../data/output/wine_metadata.ttl (7416 triples)


In [39]:
query = """
PREFIX wine: <https://wafaahs.github.io/wine-kg#>
SELECT ?wine ?variety ?points
WHERE {
    ?wine a wine:Wine ;
          wine:hasVariety ?variety ;
          wine:points ?points .
}
LIMIT 1
"""
for row in g.query(query):
    print(row)


(rdflib.term.URIRef('https://wafaahs.github.io/wine-kg#Wine_0'), rdflib.term.URIRef('https://wafaahs.github.io/wine-kg#Cabernet_Sauvignon'), rdflib.term.Literal('96', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))


In [49]:
query_meta = """
PREFIX meta: <https://wafaahs.github.io/wine-kg/meta#>
PREFIX wine: <https://wafaahs.github.io/wine-kg#>

SELECT ?wine
WHERE {
    ?wine meta:hasMissingField "country" .
}
"""
print("Wines missing country info:")
for row in g_meta.query(query_meta):
    print(row[0])


Wines missing country info:
https://wafaahs.github.io/wine-kg#Wine_1133
https://wafaahs.github.io/wine-kg#Wine_1440


In [52]:
import networkx as nx
from pyvis.network import Network

G = nx.Graph()
for s, p, o in g.triples((None, None, None)):
    if "Wine_" in s and ("Variety" in o or "Winery" in o):
        G.add_edge(s.split("#")[-1], o.split("#")[-1])

nt = Network(notebook=True, height="500px", width="100%")
nt.from_nx(G)
nt.show("wine_graph.html")


wine_graph.html
